In [3]:
!pip install ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.2 MB 842.9 kB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.2 MB 935.0 kB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.2 MB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.0 MB/s  0:00:01
   ---------------------------------------- 0.0/802.4 kB ? eta -:--:--
   ------------- -------------------------- 262.1/802.4 kB ? eta -:--:--
   -------------------------- ------------- 524.3/802.4 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------  786.4/802.4 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 802.4/802.4 kB 1.2 MB/s  0:00:00
   -----------------------------------

In [1]:
import torch
import cv2
import numpy as np
import pathlib

# --- FIX FOR WINDOWS ---
temp = pathlib.PosixPath
pathlib.PosixPath = pathlib.WindowsPath
# -----------------------

# ---------------- CONFIG ---------------- #
MODEL_PATH = 'best.pt' 

# Define specific thresholds for each class
# Make sure the names match exactly what your model was trained on
CUSTOM_THRESHOLDS = {
    'apple': 0.4,
    'orange': 0.01,
    'banana': 0.7   # corrected spelling from 'banna'
}

# Fallback for any object NOT listed above
DEFAULT_THRESHOLD = 0.5 
# ---------------------------------------- #

def main():
    print(f"Loading model from: {MODEL_PATH}...")
    
    try:
        model = torch.hub.load('ultralytics/yolov5', 'custom', path=MODEL_PATH, force_reload=False)
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    # Set model confidence to the LOWEST value in your list
    # This ensures the model detects everything initially, 
    # and we will filter them manually strictly afterwards.
    min_conf = min(min(CUSTOM_THRESHOLDS.values()), DEFAULT_THRESHOLD)
    model.conf = min_conf 

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print(f"Starting detection with custom thresholds: {CUSTOM_THRESHOLDS}")
    print("Press 'q' to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Inference
        results = model(frame)

        # 2. FILTERING LOGIC
        # We access the raw detections table (x1, y1, x2, y2, conf, cls)
        # and delete rows that don't meet our specific criteria.
        detections = results.xyxy[0] 
        keep_indices = []

        for i, det in enumerate(detections):
            conf = float(det[4])      # Confidence score
            cls_id = int(det[5])      # Class ID
            cls_name = model.names[cls_id] # Get the name (e.g., 'apple')

            # Determine the required threshold for this specific object
            required_conf = CUSTOM_THRESHOLDS.get(cls_name, DEFAULT_THRESHOLD)

            # Keep it only if it meets the requirement
            if conf >= required_conf:
                keep_indices.append(i)

        if len(keep_indices) > 0:
            # Overwrite the results with only the kept detections
            results.xyxy[0] = detections[keep_indices]
        else:
            # If nothing passed, clear the results
            results.xyxy[0] = torch.tensor([])

        # 3. Render
        # Now render() will only draw the boxes we kept
        annotated_frame = np.squeeze(results.render())

        # 4. Display
        cv2.imshow('YOLOv5 Custom Detection', annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()

Loading model from: best.pt...


Using cache found in C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-1-5 Python-3.13.0 torch-2.9.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Starting detection with custom thresholds: {'apple': 0.4, 'orange': 0.01, 'banana': 0.7}
Press 'q' to quit.


C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:898: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:898: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:898: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:898: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\essal/.cache\torch\hub\ultralytics_yolov5_master\models